# MuonClip angular/radial RG diagnostics — initial / best / final

This exploratory notebook uses the **same shared three-checkpoint implementation** as the canonical notebook in `notebooks/angular/07_muonclip_initial_final_angular_weightwatcher.ipynb`. It analyzes all six matrices, then filters the displayed table to `RG_MATRIX_NAME`.

The saved checkpoints are `initial`, `best`, and `final`, and the three flows are

$$initial\to best,\qquad initial\to final,\qquad best\to final.$$

Each flow is compared with an independent matched Haar/Stiefel random-angular null. No checkpoint is reconstructed and no endpoint is silently substituted.


## Angular tail and endpoint policy

The continuous projective angular coordinate is $x=(\lambda/u)/(1-\lambda/u)$. Exact or numerically indistinguishable upper-endpoint eigenvalues are **not** clipped into huge finite projective values. They are counted as discrete endpoint atoms and excluded from the continuous power-law sample. In particular, the twist endpoint $\lambda=4$ can represent a $-1$ reflection/parity eigenmode of the relative orthogonal rotation.

Every remaining positive continuous value is passed to `powerlaw.Fit(values, discrete=False, verbose=False)` with **no manual `xmin` and no `xmax`**. The package chooses the start of the tail by its standard MLE/KS search and retains all continuous observations through the largest observed value.

The analysis does not decide from alpha alone. It compares alpha, xmin, power-law KS $D$, tail count, tail length in decades, endpoint atoms, the full continuous ESD, and the conditional far tail against the random-angular null.


## Papermill usage

Environment-driven example:

```bash
cd /path/to/rg_optimizers
export RG_OPTIMIZERS_ROOT="$PWD"
export RUNROOT=/tmp/<training-run-root>
export RESULTS_ROOT="$RUNROOT/results"
export TARGET_OPTIMIZER=muon_clip
export TARGET_SEED=4242
export RUN_DIR="$RESULTS_ROOT/$TARGET_OPTIMIZER/seed_$TARGET_SEED"
export RG_MATRIX_NAME=L00_W_Q
export ANGULAR_N_NULL=500
export ANGULAR_ENDPOINT_TOL=1e-10
export ANGULAR_SHOW_PLOTS=0

papermill \
  baseline/nanogpt_one_head/notebooks/muonclip_angular_radial_rg.ipynb \
  /tmp/muonclip_angular_rg_seed${TARGET_SEED}.out.ipynb
```

Papermill `-p` overrides are also supported because configuration is constructed after Papermill injects the overridden parameter values. Optional checkpoint overrides are `INITIAL_CHECKPOINT_PATH`, `BEST_CHECKPOINT_PATH`, and `FINAL_CHECKPOINT_PATH`.


In [ ]:
import os

TARGET_SEED = int(os.environ.get("TARGET_SEED", os.environ.get("RG_SEED", "4242")))
TARGET_OPTIMIZER = os.environ.get("TARGET_OPTIMIZER", os.environ.get("OPTIMIZER_NAME", "muon_clip"))
RUN_DIR = os.environ.get("RUN_DIR", "")
RESULTS_ROOT = os.environ.get("RESULTS_ROOT", "")
RUNROOT = os.environ.get("RUNROOT", "")
INITIAL_CHECKPOINT_PATH = os.environ.get("INITIAL_CHECKPOINT_PATH", "")
BEST_CHECKPOINT_PATH = os.environ.get("BEST_CHECKPOINT_PATH", "")
FINAL_CHECKPOINT_PATH = os.environ.get("FINAL_CHECKPOINT_PATH", os.environ.get("CHECKPOINT_PATH", ""))
ANGULAR_OUTPUT_DIR = os.environ.get("ANGULAR_OUTPUT_DIR", "")
ANGULAR_N_NULL = int(os.environ.get("ANGULAR_N_NULL", "100"))
ANGULAR_N_ENTRY_NULL = int(os.environ.get("ANGULAR_N_ENTRY_NULL", "24"))
ANGULAR_MIN_TAIL = int(os.environ.get("ANGULAR_MIN_TAIL", "20"))
ANGULAR_NULL_SEED = int(os.environ.get("ANGULAR_NULL_SEED", "91337"))
ANGULAR_ENDPOINT_TOL = float(os.environ.get("ANGULAR_ENDPOINT_TOL", "1e-10"))
ANGULAR_SHOW_PLOTS = os.environ.get("ANGULAR_SHOW_PLOTS", "1")
RG_MATRIX_NAME = os.environ.get("RG_MATRIX_NAME", "L00_W_Q")


In [ ]:
from pathlib import Path
import sys

def _as_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() not in {"0", "false", "no", "off"}

def _none_if_blank(value):
    text = str(value).strip() if value is not None else ""
    return text or None

def find_experiment_root() -> Path:
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate
    raise FileNotFoundError("Set RG_OPTIMIZERS_ROOT or launch from the rg_optimizers repository")

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
if _none_if_blank(BEST_CHECKPOINT_PATH):
    os.environ["BEST_CHECKPOINT_PATH"] = str(BEST_CHECKPOINT_PATH)
else:
    os.environ.pop("BEST_CHECKPOINT_PATH", None)
os.environ["ANGULAR_ENDPOINT_TOL"] = str(ANGULAR_ENDPOINT_TOL)

from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_three_checkpoint import run_three_checkpoint_analysis

CONFIG = AnalysisConfig(
    seed=int(TARGET_SEED),
    optimizer=str(TARGET_OPTIMIZER).lower(),
    run_dir=_none_if_blank(RUN_DIR),
    results_root=_none_if_blank(RESULTS_ROOT),
    runroot=_none_if_blank(RUNROOT),
    initial_checkpoint=_none_if_blank(INITIAL_CHECKPOINT_PATH),
    final_checkpoint=_none_if_blank(FINAL_CHECKPOINT_PATH),
    output_dir=_none_if_blank(ANGULAR_OUTPUT_DIR),
    angular_nulls=int(ANGULAR_N_NULL),
    entry_nulls=int(ANGULAR_N_ENTRY_NULL),
    min_tail=int(ANGULAR_MIN_TAIL),
    null_seed=int(ANGULAR_NULL_SEED),
    show_plots=_as_bool(ANGULAR_SHOW_PLOTS),
)
print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)
print(CONFIG)
print("RG_MATRIX_NAME =", RG_MATRIX_NAME)


In [ ]:
from IPython.display import display

RESULTS, MANIFEST = run_three_checkpoint_analysis(CONFIG)
selected = RESULTS[RESULTS["matrix_name"] == str(RG_MATRIX_NAME)]

print("Checkpoints used:")
for state, path in MANIFEST["checkpoints"].items():
    print(f"  {state:7s} step={MANIFEST['steps'][state]:7d}  {path}")
if MANIFEST["best_equals_final_step"]:
    print("NOTE: best and final have the same step; best->final may be trivial.")

display(selected if len(selected) else RESULTS)
print("\nSummary CSV:", MANIFEST["summary_csv"])
print("Output directory:", MANIFEST["output_dir"])


## Interpretation

Use `initial->best` to ask whether angular organization is already present at the best-validation state, `initial->final` for the total angular flow, and `best->final` to determine whether late training continues to organize the angular sector or drifts toward the random baseline.

For the selected matrix inspect `actual_alpha`, `actual_xmin`, `actual_D`, `actual_tail_n`, `actual_tail_decades`, endpoint-atom counts, the random-null intervals, `full_continuous_ks_mc_p`, and `tail_conditional_ks_mc_p`. The conservative `candidate_nonrandom_long_tail` flag requires both a statistically nonrandom far tail and tail extent beyond the random 97.5% interval.

The primary visual diagnostics are the package-native linear/log PDF, CDF and CCDF plots; the far-tail CCDF zooms against the random-angular envelope; and the pairwise alpha, tail-length, KS-$D$, and $x_{min}$ comparison plots.
